#  Classification avec des images réelles: dogs vs cats


Le dataset complet provient d'une compétition Kaggle: https://www.kaggle.com/c/dogs-vs-cats

Le dataset en question contient 2 folders: dogs - cats

Soit un total de 2000 images.

L'objectif est de capitaliser sur les notions du cours pour développer un réseau CNN qui arrive à prédire avec la meilleure performance possible les images de chiens et de chats.

Ce notebook va permettre de structurer l'approche et la construction du modèle.

### Livrables à remettre: 

#### 1 - Ce notebook rempli avec les outputs visibles
#### 2 - Le lien URL de votre application web en section 12

## 1- Importer des librairies pertinentes:

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from matplotlib.image import imread

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


## 2- Localiser le path où se trouvent toutes les images

In [ ]:
# Define a variable as the directory path
# A adapter selon l'emplacement reel du dataset sur votre machine
my_data_dir = './data/dogs_vs_cats'


### 2.1 - Vérifier que la commande ci-dessous retourne ['train', 'validation']

In [ ]:
os.listdir(my_data_dir)

### 2.2 - Définir les variables train_path et val_path:

In [ ]:
# train and test paths (\\ for windows, / for mac)
train_path = os.path.join(my_data_dir, 'train')
test_path = os.path.join(my_data_dir, 'validation')


### 2.3 - Print le nombre d'images pour chaque class (cats & dogs) dans le dossier train et validation:

In [ ]:
# Verifier le nombre d'images de chaque classe pour le train_path et val_path
for path, name in [(train_path, "train"), (test_path, "validation")]:
    for classe in os.listdir(path):
        nb_images = len(os.listdir(os.path.join(path, classe)))
        print(f"{name} - {classe}: {nb_images} images")

# Verifier que vous avez bien 2000 images au total !
total = sum(
    len(os.listdir(os.path.join(p, c)))
    for p in [train_path, test_path]
    for c in os.listdir(p)
)
print("\nTotal images:", total)


## 3) Analyse d'exemples d'images dogs and cats

### 3.1 - Choisir au hasard une image de dog dans le train_path

In [ ]:
dog_train_dir = os.path.join(train_path, 'dog')
dog_file = random.choice(os.listdir(dog_train_dir))
dog_path = os.path.join(dog_train_dir, dog_file)
print(dog_path)


### 3.2 - Transformer cette image en numpy array

In [ ]:
dog_array = imread(dog_path)


### 3.3 - Vérifier les dimensions de cette image

In [ ]:
dog_array.shape


### 3.4 -Plot cette image via 'imshow'

In [ ]:
plt.imshow(dog_array)
plt.axis('off')
plt.title("Exemple d'image - dog")
plt.show()


### 3.5 - Refaire le même travail avec l'image d'un cat depuis le dossier train 

In [ ]:
cat_train_dir = os.path.join(train_path, 'cat')
cat_file = random.choice(os.listdir(cat_train_dir))
cat_path = os.path.join(cat_train_dir, cat_file)

cat_array = imread(cat_path)
print(cat_array.shape)

plt.imshow(cat_array)
plt.axis('off')
plt.title("Exemple d'image - cat")
plt.show()


## 4) Créer un ImageDataGenerator qui effectue un retraitement "pertinent" de ces images:

In [ ]:
# On va definir un input shape fixe pour normaliser toutes les images
image_shape = (150, 150, 3)


In [ ]:
image_generator = ImageDataGenerator(
    rescale=1/255,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
)


## 5) Construire un modèle CNN from scratch pour la classification binaire de ces images:


**- Utiliser à minima les types layers suivants: Conv2D, MaxPooling2D, Dense.**

**- Utiliser également la technique du Dropout.**

**- Prendre un input_shape arbitraire fixe et approprié**

**- Print le model summary**

**- Ne pas hésiter à ajouter des techniques ou des méthodes sur les données ou le modèle pour améliorer la performance !**

**L'objectif est de maximiser l'accuracy sur les données de test**

In [ ]:
# Visualisation de l'effet du retraitement (random_transform) sur une image
plt.imshow(image_generator.random_transform(dog_array))
plt.axis('off')
plt.title("Image apres transformation aleatoire")
plt.show()


In [ ]:
plt.imshow(image_generator.random_transform(dog_array))
plt.axis('off')
plt.title("Image apres transformation aleatoire (2)")
plt.show()


In [ ]:
plt.imshow(image_generator.random_transform(cat_array))
plt.axis('off')
plt.title("Image apres transformation aleatoire - cat")
plt.show()


In [ ]:
batch_size = 32


In [ ]:
# Create the CNN model
model = Sequential()

model.add(Conv2D(filters=32, kernel_size=(3, 3), input_shape=image_shape, activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

model.summary()


In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


### 5.2 Créer une instance de EarlyStopping

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


### 5.3 Créer un generator pour le train et validation set: 

In [ ]:
# Create the train_image_generator
train_image_generator = image_generator.flow_from_directory(
    train_path,
    target_size=image_shape[:2],
    color_mode='rgb',
    batch_size=batch_size,
    class_mode='binary'
)


In [ ]:
# Create the validation_image_generator
# Pas de data augmentation sur la validation, juste le rescale
val_datagen = ImageDataGenerator(rescale=1/255)

val_image_generator = val_datagen.flow_from_directory(
    test_path,
    target_size=image_shape[:2],
    color_mode='rgb',
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)


In [ ]:
# verify the classes dictionary (1 is dog or cat ?)
train_image_generator.class_indices
# Par convention alphabetique : cat -> 0, dog -> 1


### 5.3 Entrainer le modèle à partir du train_image_generator et utiliser le EarlyStopping

In [ ]:
# suivre a la fois la loss, accuracy, val_loss et val_accuracy
results = model.fit(
    train_image_generator,
    epochs=30,
    validation_data=val_image_generator,
    callbacks=[early_stop]
)


## 8) Evaluation du modèle

### 8.1 Sauvegarder les losses dans un dataframe

In [ ]:
losses = pd.DataFrame(model.history.history)
losses.head()


### 8.2 Plot le training et validation loss 

In [ ]:
losses[['loss', 'val_loss']].plot()
plt.title("Training vs Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

losses[['accuracy', 'val_accuracy']].plot()
plt.title("Training vs Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()


### 8.3 Calculer les probabilités pour le validation image generator 

In [ ]:
val_image_generator.reset()
y_pred_proba = model.predict(val_image_generator)


### 8.4 Transformer ces probabilités en classes en prenant un threshold de 0.5

In [ ]:
y_pred_class = (y_pred_proba > 0.5).astype(int).reshape(-1)


### 8.5 Récupérer le vecteur des true labels à partir du validation image generator

In [ ]:
y_test = val_image_generator.classes


### 8.6 Afficher le classification report et la matrice de confusion

In [ ]:
print(classification_report(y_test, y_pred_class, target_names=list(train_image_generator.class_indices.keys())))

cm = confusion_matrix(y_test, y_pred_class)
print("Confusion matrix:")
print(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title("Confusion matrix")
plt.colorbar()
classes = list(train_image_generator.class_indices.keys())
plt.xticks([0, 1], classes)
plt.yticks([0, 1], classes)
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


### 8.7 KPI final: quel est l'accuracy du model sur les données de test ? Etes-vous satisfaits de la performance de votre modèle ?

In [ ]:
final_accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy finale sur le jeu de validation: {final_accuracy:.2%}")

# Commentaire (a completer selon vos resultats) :
# Le modele from scratch obtient une accuracy correcte compte tenu du peu de donnees (2000 images),
# mais il y a probablement de la marge d'amelioration via la data augmentation et/ou le transfer learning
# (voir sections 10 et 11).


In [ ]:
# Sauvegarde du modele "from scratch"
model.save('cnn_from_scratch.keras')


# 9) Prédictions sur des cas particuliers

### 9.1 Afficher quelques images des données de test où le modèle s'est trompé.

In [ ]:
# Indices des images mal classees
misclassified_idx = np.where(y_pred_class != y_test)[0]
print(f"Nombre d'images mal classees: {len(misclassified_idx)} / {len(y_test)}")

filepaths = val_image_generator.filepaths
class_labels = {v: k for k, v in train_image_generator.class_indices.items()}


In [ ]:
n_display = min(9, len(misclassified_idx))
sample_idx = random.sample(list(misclassified_idx), n_display) if n_display > 0 else []

plt.figure(figsize=(12, 12))
for i, idx in enumerate(sample_idx):
    img = load_img(filepaths[idx], target_size=image_shape[:2])
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    true_label = class_labels[y_test[idx]]
    pred_label = class_labels[y_pred_class[idx]]
    proba = y_pred_proba[idx][0]
    plt.title(f"Vrai: {true_label} | Predit: {pred_label} ({proba:.2f})")
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Repartition des erreurs par classe reelle
errors_df = pd.DataFrame({
    'true_label': [class_labels[y] for y in y_test[misclassified_idx]],
})
print(errors_df['true_label'].value_counts())


### 9.2 Ces images ont-elles des patterns en commun ?

In [ ]:
# Observations (a completer/adapter selon vos resultats) :
# - Les erreurs concernent souvent des images avec un fond complexe ou peu lumineux,
#   plusieurs animaux dans le cadre, l'animal partiellement cache, ou un angle de prise de vue inhabituel.
# - Certaines images de petits chats/chiots peuvent etre confondues a cause d'une morphologie proche.
# - Une amelioration possible: davantage de donnees et de la data augmentation plus agressive (section 10),
#   ou l'usage de transfer learning (section 11) qui capte des features plus robustes.
print("Voir commentaires ci-dessus.")


# 10) Data augmentation (optionnel)

Utiliser des techniques de Data augmentation. L'objectif est d'enrichir le training set à partir des images initiales afin d'améliorer la performance du modèle.

Votre accuracy s'améliore t-elle post votre data augmentation ?

Vous êtes libre de structurer cette partie comme vous le jugez pertinent.

In [ ]:
# Data augmentation plus poussee
augmented_datagen = ImageDataGenerator(
    rescale=1/255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

train_image_generator_aug = augmented_datagen.flow_from_directory(
    train_path,
    target_size=image_shape[:2],
    color_mode='rgb',
    batch_size=batch_size,
    class_mode='binary'
)


In [ ]:
# On reprend une architecture similaire pour comparer equitablement
model_aug = Sequential()
model_aug.add(Conv2D(32, (3, 3), input_shape=image_shape, activation='relu'))
model_aug.add(MaxPooling2D(pool_size=(2, 2)))
model_aug.add(Conv2D(64, (3, 3), activation='relu'))
model_aug.add(MaxPooling2D(pool_size=(2, 2)))
model_aug.add(Conv2D(64, (3, 3), activation='relu'))
model_aug.add(MaxPooling2D(pool_size=(2, 2)))
model_aug.add(Flatten())
model_aug.add(Dense(128, activation='relu'))
model_aug.add(Dropout(0.5))
model_aug.add(Dense(1, activation='sigmoid'))

model_aug.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop_aug = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

results_aug = model_aug.fit(
    train_image_generator_aug,
    epochs=30,
    validation_data=val_image_generator,
    callbacks=[early_stop_aug]
)


In [ ]:
val_image_generator.reset()
y_pred_proba_aug = model_aug.predict(val_image_generator)
y_pred_class_aug = (y_pred_proba_aug > 0.5).astype(int).reshape(-1)

acc_aug = accuracy_score(y_test, y_pred_class_aug)
print(f"Accuracy avec data augmentation renforcee: {acc_aug:.2%}")
print(f"Accuracy du modele initial (rappel): {final_accuracy:.2%}")
# Comparer acc_aug a final_accuracy pour conclure si l'augmentation a aide.


# 11) Transfer learning

Utiliser la technique de transfer learning à partir d'un modèle open source qui vous semble pertinent. Justifier le choix du modèle ? 

Au final, votre accuracy s'est elle améliorée significativement ?

Vous êtes libre de structurer cette partie comme vous le jugez pertinent.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# Choix de VGG16 : modele pre-entraine sur ImageNet (1.4M images, 1000 classes dont des chiens et chats),
# architecture simple et robuste, tres utilisee comme base de transfer learning pour des taches de vision.
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=image_shape
)
conv_base.trainable = False  # On gele les poids du modele pre-entraine

conv_base.summary()


In [ ]:
transfer_model = Sequential()
transfer_model.add(conv_base)
transfer_model.add(Flatten())
transfer_model.add(Dense(128, activation='relu'))
transfer_model.add(Dropout(0.5))
transfer_model.add(Dense(1, activation='sigmoid'))

transfer_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
transfer_model.summary()

# Generators avec le preprocessing specifique a VGG16
transfer_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)
transfer_val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

transfer_train_gen = transfer_train_datagen.flow_from_directory(
    train_path, target_size=image_shape[:2], batch_size=batch_size, class_mode='binary'
)
transfer_val_gen = transfer_val_datagen.flow_from_directory(
    test_path, target_size=image_shape[:2], batch_size=batch_size, class_mode='binary', shuffle=False
)

early_stop_tl = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

results_tl = transfer_model.fit(
    transfer_train_gen,
    epochs=20,
    validation_data=transfer_val_gen,
    callbacks=[early_stop_tl]
)


In [ ]:
transfer_val_gen.reset()
y_pred_proba_tl = transfer_model.predict(transfer_val_gen)
y_pred_class_tl = (y_pred_proba_tl > 0.5).astype(int).reshape(-1)
y_test_tl = transfer_val_gen.classes

acc_tl = accuracy_score(y_test_tl, y_pred_class_tl)
print(f"Accuracy avec transfer learning (VGG16): {acc_tl:.2%}")
print(f"Accuracy du modele from scratch (rappel): {final_accuracy:.2%}")
# Le transfer learning ameliore generalement nettement l'accuracy car VGG16
# a deja appris des features visuelles riches sur un tres large dataset (ImageNet).

# On sauvegarde le meilleur modele (a utiliser dans la webapp - section 12)
transfer_model.save('best_model.keras')


# 12) Déploiement

À partir du modèle définitif que vous aurez construit et sauvegardé sur Keras, vous devez créer une webapp où l'utilisateur peut se connecter à l'URL, charger une image de chien ou chat et obtenir en retour la prédiction du modèle.

Stack recommandé: streamlit pour le développement de la webapp & render pour le hosting

In [ ]:
# Le modele final (best_model.keras) sera utilise dans l'application Streamlit.
# Voir le fichier app.py fourni avec ce projet pour le code de la webapp.
print("Modele final sauvegarde sous best_model.keras, pret pour le deploiement.")


In [ ]:
# Exemple minimal du contenu de app.py (a adapter / voir fichier separe) :
app_py_code = """
import streamlit as st
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

st.title("Classification Chiens vs Chats")

model = load_model('best_model.keras')

uploaded_file = st.file_uploader("Charger une image (chien ou chat)", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    img = load_img(uploaded_file, target_size=(150, 150))
    st.image(img, caption="Image chargee", use_column_width=True)

    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    proba = model.predict(img_array)[0][0]
    label = "Dog" if proba > 0.5 else "Cat"
    confiance = proba if proba > 0.5 else 1 - proba

    st.write(f"Prediction : **{label}** (confiance: {confiance:.2%})")
"""
print(app_py_code)


In [ ]:
# Une fois l'app.py et le modele pousses sur un repo GitHub, deployer sur Render (ou Streamlit Cloud) :
# 1. Creer un fichier requirements.txt (streamlit, tensorflow, numpy, pillow)
# 2. Pousser le code + le modele (best_model.keras) sur GitHub
# 3. Sur Render: New Web Service -> connecter le repo -> build command "pip install -r requirements.txt"
#    -> start command "streamlit run app.py --server.port $PORT --server.address 0.0.0.0"
# 4. Recuperer l'URL publique generee par Render

# URL de l'application web deployee :
url_app = "https://VOTRE-APP.onrender.com"  # <-- a remplacer par votre URL reelle
print("URL de la webapp:", url_app)
